# PySpark batching demo (50 rows)

This notebook creates a 50-row `input_df`, builds `with_groups_df` from `input_df` directly, then creates final batches.

In [ ]:
from sentence_transformers import SentenceTransformer
from pyspark.ml.feature import BucketedRandomProjectionLSH
from pyspark.sql import SparkSession
from pyspark_batching_solution import build_with_groups_df, build_query_batches

spark = SparkSession.builder.appName('query-batching-demo').getOrCreate()

In [ ]:
rows = [
    ('iphone', 'PH', 'en'), ('i-phone', 'PH', 'en'), ('I phone', 'PH', 'en'), ('apple iphone', 'PH', 'en'), ('new iphone', 'PH', 'en'),
    ('iphone 13', 'PH', 'en'), ('iphone 14', 'PH', 'en'), ('iphone pro', 'PH', 'en'), ('iphone max', 'PH', 'en'), ('apple phone', 'PH', 'en'),
    ('samsung s24', 'PH', 'en'), ('samsung s24 ultra', 'PH', 'en'), ('s24 ultra', 'PH', 'en'), ('samsung galaxy s24', 'PH', 'en'), ('galaxy s24', 'PH', 'en'),
    ('s24', 'PH', 'en'), ('s24 plus', 'PH', 'en'), ('s24 5g', 'PH', 'en'), ('galaxy s24 ultra', 'PH', 'en'), ('samsung phone s24', 'PH', 'en'),
    ('google pixel', 'PH', 'en'), ('pixel by google', 'PH', 'en'), ('pixel 8', 'PH', 'en'), ('google pixel 8', 'PH', 'en'), ('pixel phone', 'PH', 'en'),
    ('pixel 8 pro', 'PH', 'en'), ('pixel google phone', 'PH', 'en'), ('google phone pixel', 'PH', 'en'), ('new pixel', 'PH', 'en'), ('pixel android', 'PH', 'en'),
    ('moto 5g', 'PH', 'en'), ('motorola 5g', 'PH', 'en'), ('moto g 5g', 'PH', 'en'), ('motorola phone', 'PH', 'en'), ('moto g', 'PH', 'en'),
    ('xiaomi redmi', 'PH', 'en'), ('redmi phone', 'PH', 'en'), ('redmi note', 'PH', 'en'), ('xiaomi phone', 'PH', 'en'), ('mi phone', 'PH', 'en'),
    ('oppo reno', 'PH', 'en'), ('oppo phone', 'PH', 'en'), ('reno oppo', 'PH', 'en'), ('oppo 5g', 'PH', 'en'), ('oppo mobile', 'PH', 'en'),
    ('vivo v30', 'PH', 'en'), ('vivo phone', 'PH', 'en'), ('vivo 5g', 'PH', 'en'), ('v30 vivo', 'PH', 'en'), ('new vivo', 'PH', 'en')
]

input_df = spark.createDataFrame(rows, ['query', 'location', 'language'])
input_df.show(50, truncate=False)

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

lsh_model = BucketedRandomProjectionLSH(
    inputCol='features',
    outputCol='hashes',
    bucketLength=1.5,
    numHashTables=3
)

In [ ]:
with_groups_df = build_with_groups_df(
    input_df=input_df,
    model=model,
    lsh_model=lsh_model,
    threshold=0.25
)

with_groups_df.orderBy('final_group', 'query').show(100, truncate=False)

In [ ]:
final_df = build_query_batches(with_groups_df, batch_size=5)
final_df.show(100, truncate=False)